### Query Rewriting with Keyword Dictionary

In [ ]:
# pull text embedding model (text to vector)
# !ollama pull nomic-embed-text

In [ ]:
# 3: Embedding model
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="nomic-embed-text")

# dimension 확인용 test
# test = embedding.embed_query("test")
# print(len(test))  # output을 pinecone index의 dimensions에 넣어

In [ ]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-ollama \
#     langchain

In [ ]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# pinecone 연결
load_dotenv()
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# pinecone index에 연결
database = PineconeVectorStore.from_existing_index(
    index_name="tax-markdown-index", embedding=embedding
)

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

In [ ]:
# Ollama LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest")

In [ ]:
# RetrievalQA Chain
# %pip install -U langchain langchainhub -q

In [ ]:
from langchain_classic import hub

# get prompt from hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# define dictionary
dictionary = ["사람을 나타내는 표현 -> 거주자"]

dictionary_prompt = ChatPromptTemplate.from_template(
    f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서, 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    변경된 질문만 출력해주세요. 다른 설명은 하지 마세요.

    사전: {dictionary}

    질문: {{question}}
"""
)

dictionary_chain = dictionary_prompt | llm | StrOutputParser()

In [ ]:
# new query generated from dictionary_chain
query = dictionary_chain.invoke({"question": query})

In [ ]:
query

In [ ]:
# retrieve document based on query
retriever = database.as_retriever(search_kwargs={"k": 4})
retriever.invoke(query)

In [ ]:
# Build QA Chain
from langchain_classic.chains import RetrievalQA

# define QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=retriever, chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain.invoke({"query": query})

In [ ]:
ai_message

In [ ]:
# build tax_chain (dictionary_chain + qa_chain)
tax_chain = {"query": dictionary_chain} | qa_chain

### test tax_chain (final_chain) with new query

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

In [ ]:
ai_response = tax_chain.invoke({"question": query})

In [ ]:
ai_response